In [5]:
from new_pack import triton_quantize_and_pack_along_last_dim
# def triton_quantize_and_pack_along_last_dim(data: torch.Tensor, group_size: int, bit: int):
# 	assert len(data.shape) == 4
# 	shape = data.shape
# 	B, nh, D, T = shape
# 	# ================== Get Scale & Zeros ===============
# 	assert T % group_size == 0
# 	num_groups = T // group_size
# 	new_shape = (B * nh * D, num_groups, group_size)
# 	scale_mn_shape = B, nh, D, num_groups
# 	# Quantize
# 	data = data.reshape(new_shape)
# 	mx = torch.empty((B * nh * D, num_groups), device=data.device, dtype=data.dtype)
# 	mn = torch.empty((B * nh * D, num_groups), device=data.device, dtype=data.dtype)
# 	BLOCK_SIZE_N = 128
# 	grid = lambda meta: (triton.cdiv(data.shape[0]*data.shape[1], BLOCK_SIZE_N),)
# 	_minmax_along_last_dim[grid](data, mn, mx,
# 							 data.numel(), data.shape[0], num_groups, group_size,
# 							 BLOCK_SIZE_N=BLOCK_SIZE_N, num_warps=8) 
# 	# mn = torch.min(data, dim=-1, keepdim=True)[0].squeeze(-1)
# 	# mx = torch.max(data, dim=-1, keepdim=True)[0].squeeze(-1)
# 	scale = (mx - mn) / (2 ** bit - 1)
# 	data = data - mn.unsqueeze(-1)
# 	data.div_(scale.unsqueeze(-1))
# 	data = data.clamp_(0, 2 ** bit - 1).round_().to(torch.int32)
# 	data = data.view(-1, T)
# 	feat_per_int = 32 // bit
# 	packshape = (np.prod(shape[:-1]), shape[-1] // feat_per_int,)
# 	code = torch.zeros(*packshape, device=data.device, dtype=torch.int32)
# 	grid = lambda meta: (triton.cdiv(data.shape[0], BLOCK_SIZE_N), data.shape[1] // feat_per_int,)
# 	_pack_along_last_dim[grid](bit, data, code, data.shape[0], 
# 								data.shape[1], feat_per_int, 
# 								BLOCK_SIZE_N=BLOCK_SIZE_N, 
# 								num_warps=8)
# 	return code.view(B, nh, D, -1), scale.reshape(scale_mn_shape), mn.reshape(scale_mn_shape)

In [6]:
import torch
test_matrix = torch.randn((2, 8, 128, 32)).to('cuda:0')

In [7]:
quantized_matrix, scale, mn = triton_quantize_and_pack_along_last_dim(test_matrix,32,4)

In [8]:
quantized_matrix.shape

torch.Size([2, 8, 128, 4])

In [9]:
scale.shape

torch.Size([2, 8, 128, 1])

In [20]:
def simquant(inp, bits, q_channel=-1):
    if bits == -16:
        return inp
    
    maxval = torch.max(inp, dim=q_channel).values
    minval = torch.min(inp, dim=q_channel).values
    # print('inp', inp.shape)


    rangeval = (maxval - minval) / 1.
    qx = (2**bits - 1) / rangeval
    offset = minval * qx
    offset = offset.unsqueeze(q_channel)
    qx = qx.unsqueeze(q_channel)
    # print('qx', qx.shape)
    # print('offset', offset.shape)
    qinp = torch.round(qx * inp - offset)
    print(qinp.max())
    qinp = torch.clip(qinp, min=0, max=2**bits - 1)
    qinp_out = (qinp + offset) / qx
    qinp_out = torch.nan_to_num(qinp_out, nan=0.0, posinf=0.0, neginf=0.0)
    return qinp_out, minval.unsqueeze(q_channel), qx

In [21]:
_, mn_, scale_ = simquant(test_matrix, 4, -1)

tensor(15., device='cuda:0')


In [30]:
(scale_).min()

tensor(2.1213, device='cuda:0')

In [28]:
scale.max()

tensor(0.4714, device='cuda:0')

In [2]:
import torch
a = torch.tensor([1.2], dtype=torch.float16)

In [17]:
a = torch.randn((2,4),dtype=torch.float16)
a[0,0]

tensor(0.1891, dtype=torch.float16)

In [18]:
b = a/torch.tensor(1000000.0)
b = b*torch.tensor(1000000.0)
b[0,0]

tensor(0.1788, dtype=torch.float16)